In [1]:
import pandas as pd
import duckdb

pd.set_option('display.max_rows', 20)      # shows ~first 10 + last 10
pd.set_option('display.min_rows', 30) 

data_dir="../csv"

# create functions!

In [2]:
def lefty(self, n=10):
    
    if len(self) <= n:
        df2 = self
    else:
        head = self.head(n)
        tail = self.tail(n)
        ellipsis_row = pd.DataFrame([['...'] * len(self.columns)], columns=self.columns)
        df2 = pd.concat([head, ellipsis_row, tail], ignore_index=True)

    text_cols = self.select_dtypes(include='object').columns
    
    if len(text_cols) > 0:
        return df2.style.set_properties(**{'text-align': 'left'}, subset=text_cols)
    else:
        return df2



################################################################

def show_all(self):
    with pd.option_context('display.max_rows', len(self)):
        display(self)

################################################################

pd.DataFrame.lefty = lefty
pd.DataFrame.show_all = show_all

# Make monthly sales summary.

## load parts.csv 

In [3]:
parts = pd.read_csv(f"{data_dir}/parts.csv")
parts['SQFT'] = parts['SQFT'].astype('Int64')
parts.lefty()

,PART_ID,PROD,PART_TYPE,SQFT
0,tin-clip,tin-roof,nan,10
1,tin-decoration,tin-roof,nan,
2,gold-clip,golf-roof,nan,50
3,gold-decoration,golf-roof,nan,
4,glass-clip,glass-roof,new,100
5,glass-decoration,glass-roof,new,
6,roof-polish,nan,nan,


## Load sales.csv 

- Test data generated by script Grok made.
- 2 suppliers with 2 licensees
- Plus another licensee, who buys from both suppliers.
- Date range 3/2023 to 1/2026

In [4]:

sales = pd.read_csv(f"{data_dir}/sales-grok.csv", parse_dates=['DATE'])
sales['AMT'] = sales['AMT'].astype('int64')
sales['QTY'] = sales['QTY'].astype('int64')
sales.lefty()

,DATE,SUPPLIER,LICENSEE,PO,PART_ID,QTY,AMT
0,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,roof-polish,57,11100
1,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,glass-decoration,114,86100
2,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,gold-clip,156,89500
3,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,glass-clip,109,128600
4,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,gold-decoration,165,64400
5,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,tin-decoration,44,3800
6,2023-03-08 00:00:00,nippon-metal,ninja-roofing,PO0001,tin-clip,176,16000
7,2023-03-04 00:00:00,nippon-metal,rice-roofers,PO0002,gold-clip,78,35300
8,2023-03-04 00:00:00,nippon-metal,rice-roofers,PO0002,tin-decoration,95,8900
9,2023-03-04 00:00:00,nippon-metal,rice-roofers,PO0002,gold-decoration,119,57800


# Calculate monthly sales

In [7]:
duckdb.query("DROP VIEW IF EXISTS monthly_sales");

duckdb.query("""
    CREATE VIEW monthly_sales AS
    SELECT 
        strftime('%Y-%m', date) AS month,
        supplier,
        licensee,
        part_id,
        CAST(SUM(qty) AS INT64) AS qty, 
        CAST(SUM(amt) AS INT64) AS amt
    FROM sales
    GROUP BY month, supplier, licensee, part_id
    ORDER BY month, supplier, licensee, part_id;
""")

 
#duckdb.query("SELECT * FROM monthly_sales").df().show_all()#.lefty()
duckdb.query("SELECT * FROM monthly_sales").df().lefty()

,month,SUPPLIER,LICENSEE,PART_ID,qty,amt
0,2023-03,nippon-metal,ninja-roofing,glass-clip,109,128600
1,2023-03,nippon-metal,ninja-roofing,glass-decoration,114,86100
2,2023-03,nippon-metal,ninja-roofing,gold-clip,156,89500
3,2023-03,nippon-metal,ninja-roofing,gold-decoration,165,64400
4,2023-03,nippon-metal,ninja-roofing,roof-polish,57,11100
5,2023-03,nippon-metal,ninja-roofing,tin-clip,176,16000
6,2023-03,nippon-metal,ninja-roofing,tin-decoration,44,3800
7,2023-03,nippon-metal,rice-roofers,glass-clip,158,186200
8,2023-03,nippon-metal,rice-roofers,gold-clip,78,35300
9,2023-03,nippon-metal,rice-roofers,gold-decoration,119,57800


In [11]:
duckdb.query("DROP VIEW IF EXISTS monthly_sales_ytd");

duckdb.query("""
CREATE VIEW monthly_sales_ytd AS
SELECT
    month,
    supplier,
    printf('%,d', SUM(amt)) AS month_amt,
    printf('%,d', SUM(SUM(amt)) OVER (
        PARTITION BY supplier, SUBSTR(month,1,4)
        ORDER BY month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )) AS ytd_amt
FROM monthly_sales
GROUP BY month, supplier
ORDER BY supplier, month
""")

duckdb.query("select * from monthly_sales_ytd").df().lefty()

,month,SUPPLIER,month_amt,ytd_amt
0,2023-03,nippon-metal,"709,400","709,400"
1,2023-04,nippon-metal,"260,900","970,300"
2,2023-05,nippon-metal,"629,100","1,599,400"
3,2023-06,nippon-metal,"1,498,200","3,097,600"
4,2023-07,nippon-metal,"1,352,400","4,450,000"
5,2023-08,nippon-metal,"1,146,500","5,596,500"
6,2023-09,nippon-metal,"1,343,300","6,939,800"
7,2023-10,nippon-metal,"742,500","7,682,300"
8,2023-11,nippon-metal,"918,100","8,600,400"
9,2023-12,nippon-metal,"760,300","9,360,700"


In [ ]:
# version 2 of ytd view
duckdb.query("DROP VIEW IF EXISTS monthly_sales_ytd");

duckdb.query("""
CREATE VIEW monthly_sales_ytd AS
SELECT
    month,
    supplier,
    printf('%,d', SUM(amt)) AS month_amt,
    printf('%,d', SUM(SUM(amt)) OVER (
        PARTITION BY supplier, SUBSTR(month,1,4)
        ORDER BY month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )) AS ytd_amt
FROM monthly_sales
GROUP BY month, supplier
ORDER BY supplier, month
""")

duckdb.query("select * from monthly_sales_ytd").df()

In [ ]:
duckdb.query("""
    SELECT 
        strftime('%Y-%m', date) AS month,
        licensee,
        COUNT(DISTINCT po) AS po_count
    FROM sales
    GROUP BY month, licensee
    ORDER BY month, licensee
""").df()


In [ ]:
df = duckdb.query("""
    SELECT 
        strftime('%Y-%m', date) AS month,
        supplier,
        printf('%,d', SUM(amt)) as tot_amt

    FROM sales
    GROUP BY month, supplier
    ORDER BY month, supplier
""").df()

print("\n\nSupplier Monthly Totals - simple SQL query\n")

df